Here we combine the three recommendation system (the topic based recommender, ..., ... ) to make a combined recommendation system.

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


First we connect to the database and get the events (ID, title, description) and the events users betted on.

In [9]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    """Remove the schema parameter from DATABASE_URL for asyncpg"""
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    # Remove 'schema' parameter if it exists
    query_params.pop('schema', None)

    # Rebuild the URL without schema parameter
    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

Here we make the embeddings for the topic-based recommender.

In [10]:
import pandas as pd
import faiss
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# fetch all rows and convert to DataFrame
event_rows = await conn.fetch('SELECT id, title, description FROM "Event";')
events_df = pd.DataFrame([dict(r) for r in event_rows])
print(f'Loaded {len(events_df)} rows into comment_df')
events_df.set_index('id', inplace=True)

topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2")

# fetch all rows and convert to DataFrame
user_event_associations = await conn.fetch('SELECT * FROM user_event_associations;')
user_event_associations_df = pd.DataFrame([dict(r) for r in user_event_associations])

# Create events_df from user_event_associations_df
# Extract unique events with their titles and descriptions
events_from_users = user_event_associations_df[['event_id', 'event_title', 'event_description']].drop_duplicates()

# Rename columns to match original events_df structure
events_from_users = events_from_users.rename(columns={
    'event_id': 'id',
    'event_title': 'title',
    'event_description': 'description'
})

# Convert id to int and set as index
events_from_users['id'] = events_from_users['id'].astype(int)
events_from_users = events_from_users.set_index('id')


# Load the same embedding model used by topic_model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# Concatenate the dataframes
# Use concat to combine them, keeping only events that don't already exist in events_df
combined_events = pd.concat([events_df, events_from_users])

# Remove duplicates (keep first occurrence - from original events_df)
combined_events = combined_events[~combined_events.index.duplicated(keep='first')]


# Update events_df to the combined version
events_df = combined_events

# Create combined text for embeddings
events_df['combined_text'] = events_df['title'].fillna('') + " " + events_df['description'].fillna('')

embeddings = embedding_model.encode(events_df['combined_text'].tolist(), show_progress_bar=True)

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
index.add(normalized_embeddings.astype('float32'))

index_to_id = {i: id for i, id in enumerate(events_df.index)}
id_to_index = {id: i for i, id in enumerate(events_df.index)}


Loaded 65056 rows into comment_df


Here is our combined recommender:

In [3]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based

def combined_recommendation_for_user(user_address):
    # This is the topic based recommender
    top_n=10
    similar_per_event=5
    topic_based_recommendations= recommend_events_for_user_by_topic_based(id_to_index, normalized_embeddings,index,index_to_id, user_event_associations_df,events_df, user_address, top_n, similar_per_event)
    return topic_based_recommendations

In [18]:
example_user = user_event_associations_df.iloc[0]['address']
recommendations= combined_recommendation_for_user(example_user)
if recommendations:
    for rank, rec in enumerate(recommendations, 1):
        print(f"{rank}. [{rec['id']}] {rec['title']}")

1. [57723] 2nd Largest company end of 2025?
2. [61883] Israel strikes Gaza by...?
3. [45221] Trump strikes another drug boat by Sep 30?
4. [64121] #2 Searched Person on Google this year?
5. [41835] When will Israel raid Gaza aid flotilla?
6. [37155] Will Russia capture Myrnohrad by August 31?
7. [36109]  Will Trump meet with Putin by August 15?
8. [51135] Will Russia capture Stepanivka by October 31?
9. [12869] NYC mayoral special election in 2024?
10. [20763] Will Ukraine agree to cede territory to Russia?
